# 01 · Ingestão Bronze (CineData Analytics)

Leitura pura dos 5 arquivos CSV brutos da *Landing Zone* (`/Volumes/workspace/default/landing/cinedata`) e gravação no formato Delta Lake no schema `cinedata_bronze`, sem aplicar transformações nas colunas originais.

**Requisitos atendidos:**
- Adição da coluna metadata `ingestion_datetime` com o timestamp da inserção.
- Gravação das tabelas em formato Delta utilizando o modo `append`.

### 🔗 0. Importação das Configurações de Ambiente
A instrução `%run ./00_Organizacao_do_Ambiente` executa o notebook de configuração inicial. 
Ela permite herdar todas as **variáveis globais** (como `landing_path`, `catalog` e os schemas `cinedata_bronze`, `cinedata_silver`, `cinedata_gold`), garantindo que este notebook utilize os caminhos e bases de dados corretos sem necessidade de redefini-los.

In [0]:
%run ./00_Organizacao_do_Ambiente

# 00 · Organização do Ambiente
### Projeto Prático — Arquitetura Medalhão com a base CineData (TMDB/IMDb)

Objetivos deste notebook:
1. Entender a estrutura de diretórios/tabelas que vamos usar (Landing → Bronze → Silver → Gold)
2. Criar o `catalog` e os `schemas` no Unity Catalog
3. Criar o **Volume** que representa a nossa zona de *Landing*
4. Validar que os arquivos `.csv` de origem do acervo de filmes estão disponíveis

Este notebook é reaproveitado pelos demais (`%run ./00_Organizacao_do_Ambiente`), então evite alterar os nomes das variáveis abaixo sem ajustar os outros notebooks.


In [0]:
from pyspark.sql.functions import current_timestamp #importar da função explicitamente no início 

## 1. Parâmetros do ambiente, Criação do Catalog, Schemas e Volume

Catalog, schemas com prefixo 'cinedata_' e volume criados/verificados com sucesso!
Bronze: workspace.cinedata_bronze
Silver: workspace.cinedata_silver
Gold:   workspace.cinedata_gold


%md
## 3. Upload dos arquivos de origem

Vamos importar os **5 arquivos** que serão utilizados no Databricks (projeto CineData) em **Catalog** → `workspace` → `cinedata_bronze` → `landing` → **Upload to this volume**[cite: 5, 6, 7]:
   - `credits_and_tags_IMDB_TMDB.csv`
   - `movies_financials_IMDB_TMDB.csv`
   - `movies_info_TMDB_IMDB.csv`
   - `movies_metrics_IMDB_TMDB.csv`
   - `movies_reviews.csv`



## 1. Mapeamento e Leitura dos Arquivos Brutos (Landing Zone)

Nesta etapa, realizamos o mapeamento dos caminhos absolutos dos **5 arquivos CSV de origem do acervo CineData Analytics** diretamente no nosso Volume montado na *Landing Zone* (`/Volumes/workspace/default/landing/cinedata`).

A leitura é executada utilizando o recurso de **inferência de esquema** (`inferSchema=True`) e preservação de cabeçalhos (`header=True`), garantindo a **leitura pura dos dados brutos** sem aplicar qualquer tipo de transformação de negócio ou alteração de colunas nesta camada.

---

**DataFrames carregados em memória:**
- `df_credits_raw`: Elenco, equipe técnica e tags (`credits_and_tags_IMDB_TMDB.csv`)
- `df_financials_raw`: Orçamento, receita e métricas financeiras (`movies_financials_IMDB_TMDB.csv`)
- `df_info_raw`: Informações cadastrais e metadados dos filmes (`movies_info_TMDB_IMDB.csv`)
- `df_metrics_raw`: Popularidade e notas médias (`movies_metrics_IMDB_TMDB.csv`)
- `df_reviews_raw`: Avaliações e resenhas dos usuários (`movies_reviews.csv`)

In [0]:
# 1. Mapeamento dos caminhos exatos na Landing Zone
path_credits = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"

# 2. Leitura pura dos arquivos CSV
df_credits_raw = spark.read.csv(path_credits, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(path_financials, header=True, inferSchema=True)
df_info_raw = spark.read.csv(path_info, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_reviews_raw = spark.read.csv(path_reviews, header=True, inferSchema=True)


## 2. Inserção do Metadata e Gravação no Delta Lake (Modo Append)

Nesta etapa, adicionamos a coluna de controle **`ingestion_datetime`** utilizando a função `current_timestamp()` para registrar o exato momento da ingestão. 

Em seguida, gravamos os DataFrames no **Delta Lake** no modo **`append`** dentro do schema **`cinedata_bronze`**, garantindo o armazenamento nativo e incremental sem alterar a estrutura dos dados originais.

In [0]:
# 4. Adição da coluna ingestion_datetime e gravação em Delta na cinedata_bronze com modo "append"

# Tabela 1: credits_and_tags
df_credits_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

# Tabela 2: movies_financials
df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

# Tabela 3: movies_info
df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_info")

# Tabela 4: movies_metrics
df_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

# Tabela 5: movies_reviews
df_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

# Validação
display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-20T23:15:50.326Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-20T23:15:50.326Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-20T23:15:50.326Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-20T23:15:50.326Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-20T23:15:50.326Z



## 3. Ingestão via API — Cotação do Dólar (Banco Central do Brasil)

Nesta etapa, atendemos ao requisito de negócio para disponibilizar a cotação do dólar comercial (BRL/USD) na camada Bronze.

---

### 📌 Diretrizes de Implementação:
1. **Parâmetros Dinâmicos (Widgets):** Utilização de `dbutils.widgets` para definição do período (`data_inicio` e `data_fim`) no formato `MM-DD-AAAA`. Por padrão, é calculada uma janela de **7 dias corridos** a partir da data de execução para garantir a captura de dias úteis (já que a PTAX não possui publicação em fins de semana e feriados).
2. **Origem:** Requisição HTTP para a API PTAX do Banco Central do Brasil (retorno em JSON).
3. **Mecanismo de Resiliência (`try-except`):** Tenta realizar a chamada HTTP direta à API do BCB. Caso o cluster (ex: ambiente Serverless ou regras de firewall corporativo) restrinja chamadas HTTP de saída ou resolução de DNS (`NameResolutionError`), o código intercepta a exceção e gera um registro de contingência para criar e manter a estrutura da tabela no Delta Lake.
4. **Destino:** Gravação no formato **Delta Lake** no modo **`append`** com a inclusão da coluna de metadados **`ingestion_datetime`** na tabela `workspace.cinedata_bronze.tb_cotacao_dolar`.


In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Configuração dos Widgets do Databricks para entrada de parâmetros
data_hoje = datetime.now()
data_7_dias_atras = data_hoje - timedelta(days=7)

default_data_inicio = data_7_dias_atras.strftime("%m-%d-%Y")
default_data_fim = data_hoje.strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", default_data_inicio, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", default_data_fim, "Data Fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# 2. Montagem da URL da API do Banco Central
url_bcb = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# Schema esperado para a tabela de cotação
schema_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

# 3. Tenta conectar na API do BCB
try:
    response = requests.get(url_bcb, timeout=10)
    
    if response.status_code == 200:
        dados_json = response.json().get('value', [])
        if len(dados_json) > 0:
            df_cotacao_raw = spark.createDataFrame(dados_json, schema=schema_cotacao)
            print(f"[OK] Cotações reais obtidas via API com sucesso ({len(dados_json)} registros).")
        else:
            raise Exception("API respondeu sem registros no período informado.")
    else:
        raise Exception(f"Erro HTTP na API: {response.status_code}")

except Exception as e:
    print(f"[AVISO] Sem acesso direto à internet no cluster ou API indisponível ({e}).")
    print("[INFO] Gerando dados de contingência (fallback) para criação da tabela Bronze...")
    
    # Dados fictícios/fallback para permitir a gravação e evolução do projeto
    dados_fallback = [
        {"dataHoraCotacao": f"{datetime.now().strftime('%Y-%m-%d')} 13:00:00.000", "cotacaoCompra": 5.25}
    ]
    df_cotacao_raw = spark.createDataFrame(dados_fallback, schema=schema_cotacao)

# 4. Inserção do metadado ingestion_datetime e gravação no Delta Lake (modo append)
df_cotacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").limit(5))

[AVISO] Sem acesso direto à internet no cluster ou API indisponível (HTTPSConnectionPool(host='olinda.bcb.gov.br', port=443): Max retries exceeded with url: /olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-13-2026'&@dataFinalCotacao='09-20-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f56c01ec7d0>: Failed to resolve 'olinda.bcb.gov.br' ([Errno -3] Temporary failure in name resolution)"))).
[INFO] Gerando dados de contingência (fallback) para criação da tabela Bronze...


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2016-09-21 00:00:00,3.2402,2026-09-21T03:09:45.864Z
2016-09-22 00:00:00,3.2009,2026-09-21T03:09:45.864Z
2016-09-23 00:00:00,3.2236,2026-09-21T03:09:45.864Z
2016-09-26 00:00:00,3.2394,2026-09-21T03:09:45.864Z
2016-09-27 00:00:00,3.2352,2026-09-21T03:09:45.864Z


### ⚠️ Nota de Arquitetura e Contingência (Ingestão de Cotações do Dólar)

> **Contexto da Implementação:**
> O ambiente *Databricks Community Edition* (gratuito) possui restrições de firewall de rede que bloqueiam chamadas de saída HTTP/API externas (`outbound network requests`), impossibilitando a execução direta da célula de requisição acima.
>
> **Estratégia de Ingestão Adotada (Workaround):**
> Para garantir a integridade dos dados e o preenchimento do histórico temporal completo, os dados oficiais fornecidos pela API do Banco Central do Brasil foram extraídos e carregados diretamente no Volume de Landing do Unity Catalog (`/Volumes/workspace/default/landing/cotacao_dolar.json`).
> 
> A célula a seguir realiza a leitura desse contrato oficial em JSON, garantindo a mesma estrutura de esquema e tipos para a construção da camada Bronze e posterior aplicação da técnica de *Forward Fill* na camada Silver.

In [0]:
import os
from pyspark.sql.functions import col, to_timestamp, current_timestamp, explode

# 1. Caminho do arquivo JSON no Volume landing
caminho_volume = "/Volumes/workspace/default/landing/cinedata/"
arquivo_json = os.path.join(caminho_volume, "cotacao_dolar.json")

# 2. Leitura do arquivo JSON
df_raw = spark.read.option("multiline", "true").json(arquivo_json)

# Caso o JSON venha dentro do padrão Olinda/OData (chave 'value')
if "value" in df_raw.columns:
    df_json = df_raw.select(explode(col("value")).alias("item")).select("item.*")
else:
    df_json = df_raw

# 3. Mapeamento direto das colunas 'data' e 'valor' para a Bronze
df_bronze_cotacao = df_json.select(
    to_timestamp(col("data"), "dd/MM/yyyy").cast("string").alias("dataHoraCotacao"),
    col("valor").cast("double").alias("cotacaoCompra")
).filter(col("dataHoraCotacao").isNotNull()) \
 .withColumn("ingestion_datetime", current_timestamp())

# 4. Salvamento na Tabela Bronze
df_bronze_cotacao.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.cinedata_bronze.tb_cotacao_dolar")

print("[OK] Tabela Bronze 'tb_cotacao_dolar' recarregada com sucesso!")

[OK] Tabela Bronze 'tb_cotacao_dolar' recarregada com sucesso!
